### Download Datasets from THE LOCALISED DSP

#### Load required libraries

In [1]:
from zoomin_client import client

#### LOCALISED-Datasharing-API-Client

https://github.com/FZJ-IEK3-VSA/LOCALISED-Datasharing-API-Client 

#### Get data

In [9]:
# download_non_sector_data.py
import time
from pathlib import Path
import pandas as pd

# ---------- Paths ----------
OUTPUT_DIR = Path("../Initial data/Non sector data")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("Code and data/Initial data/Non sector data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving data to:", OUTPUT_DIR.resolve())

# ---------- Inputs ----------
EU27 = [
    "at","be","bg","hr","cy","cz","dk","ee","fi","fr","de","el",
    "hu","ie","it","lv","lt","lu","mt","nl","pl","pt","ro","sk",
    "si","es","se"
]

DATA_CATEGORIES = {
    "Unemployment_REGIO": {
        "variables": ["percentage_of_unemployed_people"],
        "excel_file": OUTPUT_DIR / "LABOUR-Unemployment.xlsx",
    },
    "Wages_MANUF": {
        "variables": ["wages_in_manufacturing"],
        "excel_file": OUTPUT_DIR / "LABOUR-Wages.xlsx",
    },
    "Capital_Stock_Based_Prod_MANUF": {
        "variables": ["capital_stock_based_productivity_in_manufacturing"],
        "excel_file": OUTPUT_DIR / "FINANCE-Capital_Stock_Based_Prod.xlsx",
    },
}

# ---------- Helpers ----------
def clean_sheet_name(name: str, used: set) -> str:
    """Excel sheet name: max 31 chars, no []:*?/\ and must be unique."""
    bad = '[]:*?/\\'
    for ch in bad:
        name = name.replace(ch, "_")
    name = (name or "sheet")[:31]
    base, i = name, 1
    while name in used:
        suf = f"_{i}"
        name = base[: 31 - len(suf)] + suf
        i += 1
    used.add(name)
    return name

def ensure_client():
    try:
        return client  # provided by caller/environment
    except NameError:
        raise RuntimeError(
            "Define `client` before running. Expected method:\n"
            "client.get_variable_data(country_code, variable, spatial_resolution='NUTS2', result_format='df')"
        )

# ---------- Main ----------
def main():
    cli = ensure_client()

    for cat_name, meta in DATA_CATEGORIES.items():
        print(f"\n=== {cat_name} ===")
        variables = meta["variables"]
        excel_path = meta["excel_file"]

        # Collect frames per variable
        by_var = {v: [] for v in variables}

        for cc in EU27:
            for var in variables:
                try:
                    print(f"Fetching {cc.upper()} · {var} ...")
                    df = cli.get_variable_data(
                        country_code=cc,
                        variable=var,
                        spatial_resolution="NUTS2",
                        result_format="df",
                    )
                    if "country" not in df.columns:
                        df["country"] = cc.upper()
                    by_var[var].append(df)
                    time.sleep(0.2)  # be gentle
                except Exception as e:
                    print(f"  -> Error {cc}-{var}: {e}")

        # Write one Excel per category
        used_names = set()
        wrote_any = False
        with pd.ExcelWriter(excel_path, mode="w") as writer:
            for var, frames in by_var.items():
                if not frames:
                    continue
                combined = pd.concat(frames, ignore_index=True)
                # Optional: sort if you have these columns
                for cols in (["country","year"], ["NUTS_ID","year"], ["geo","time"]):
                    if all(c in combined.columns for c in cols):
                        combined = combined.sort_values(cols)
                        break
                sheet = clean_sheet_name(var, used_names)
                combined.to_excel(writer, sheet_name=sheet, index=False)
                wrote_any = True

        if wrote_any:
            print(f"✅ Saved: {excel_path.resolve()}")
        else:
            print(f"⚠️ No data written for {cat_name} (no successful downloads).")

if __name__ == "__main__":
    main()
